In [7]:
import torch
import numpy as np
from transformers import Qwen2VLProcessor
from qwen_vl_utils import process_vision_info

/hdd/Documents/Projects/miniLAVAD/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
# 1. Initialize the Processor
# This downloads the tokenizer and image processing configuration
model_id = "Qwen/Qwen2-VL-7B-Instruct"
processor = Qwen2VLProcessor.from_pretrained(model_id)

print("Processor successfully loaded!\n")

Processor successfully loaded!



In [9]:
# 2. Simulate the DataLoader Output
# We will create a dummy numpy array to represent our 10 frames
# (Format: Frames, Height, Width, RGB channels)
dummy_frames = np.zeros((10, 224, 224, 3), dtype=np.uint8) 
prompt_template = "Analyze this surveillance video and describe the anomalous event."

In [10]:
# 3. Construct the Message Dictionary
# Qwen2-VL expects a structured conversational dictionary, not raw strings.
messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "video",
                "video": dummy_frames,
                "fps": 1.0,  # We extracted 10 frames over 10 seconds (1 frame per sec)
            },
            {"type": "text", "text": prompt_template},
        ],
    }
]

In [11]:
# 4. Apply the ChatML Template
# This is the magic step where the dictionary is converted into Qwen's specific syntax
text_prompt = processor.apply_chat_template(
    messages, 
    tokenize=False, 
    add_generation_prompt=True
)

print("--- 1. RAW CHATML PROMPT ---")
print(text_prompt)

--- 1. RAW CHATML PROMPT ---
<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
<|vision_start|><|video_pad|><|vision_end|>Analyze this surveillance video and describe the anomalous event.<|im_end|>
<|im_start|>assistant



In [12]:
# 5. Process everything into PyTorch Tensors
# This handles resizing the frames and tokenizing the text
inputs = processor(
    text=[text_prompt],
    videos=[dummy_frames],
    padding=True,
    return_tensors="pt"
)

[transformers] Qwen2VL video processing does not apply the per-frame pixel cap the reference implementation (qwen-vl-utils) applies, so some videos cost far more tokens than they would there. In v5.22 the capped behavior will become the default and `cap_pixels_per_frame` will be removed. Pass `cap_pixels_per_frame=True` to adopt the reference behavior now, or `False` to keep the current behavior and silence this warning.


In [13]:
print("\n--- 2. FINAL TENSOR SHAPES ---")
print(f"Input IDs (Text Tokens): {inputs['input_ids'].shape}")
print(f"Video Pixel Values: {inputs['pixel_values_videos'].shape}")
print(f"Video Grid Thw (Temporal/Height/Width): {inputs['video_grid_thw'].shape}")


--- 2. FINAL TENSOR SHAPES ---
Input IDs (Text Tokens): torch.Size([1, 353])
Video Pixel Values: torch.Size([1280, 1176])
Video Grid Thw (Temporal/Height/Width): torch.Size([1, 3])
